# Data Preprocessing & Feature Encoding
#### Builds the training-ready dataset from `DI_Leather_preprocessed.csv`

This notebook implements **Target & Leakage-Safe Feature Set** and **Feature Engineering** of the project plan. It takes the cleaned EDA dataset and produces `training_ready.csv`, a fully numeric, encoded dataset ready for model training in the next notebook.

**What this notebook does, in order:**
1. Load the preprocessed dataset and create the target variable (`achievement_ratio`)
2. Drop non-predictive identifier columns and all leakage columns 
3. Exclude Factory Unit 03 (insufficient sample size)
4. Encode categorical features (binary, ordinal, one-hot)
5. Export the final training-ready dataset

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("/Users/pramo/Documents/Production-optimization/data/DI_Leather_preprocessed.csv")


## 2. Target Variable: Achievement Ratio
Defined as `actual_qty / planned_qty`, per the finalized target direction.


In [2]:
df.head()

,financial_year,financial_month,week_no,week_of_month,week_start_date,week_end_date,factory_unit,product_family,item_name,size_id,...,rate,actual_qty,no_of_workers,working_hours,total_duration_per_shoe_mins,machine_capacity_per_day,machine_count,production_gap_qty,bottleneck_flag,order_category
0,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI (VAST) - BROWN,12,...,4363.35,32,128,5835.39,115.22,200,8,3,False,Others
1,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI - L - BLACK,7,...,4870.92,30,146,6669.92,110.94,200,7,11,False,Others
2,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Ladies,BINOLI - L - GREEN,9,...,7003.71,10,151,6926.29,107.55,200,8,5,False,Others
3,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Gents,BOOT HIGH LEG - BLACK,13,...,4676.91,1,128,5997.94,106.29,200,7,4,False,Forces/Police
4,2022-2023,Apr,3,3,2022-04-18,2022-04-22,Factory Unit 01,Gents,BOOT HIGH LEG - L - BLACK,13,...,3803.13,9,103,4734.43,106.48,200,8,2,False,Forces/Police


## 3. Drop Non-Predictive Identifier Columns
`financial_year`, `week_no`, `week_start_date`, `week_end_date` are dropped here. Calendar context is retained separately through `financial_month` and `week_of_month`, which are encoded later in this notebook - the raw date/year columns themselves add no direct predictive value once that context is captured.


In [3]:
columns_to_drop = [
    'financial_year',
    'week_no',
    'week_start_date',
    'week_end_date'
]

# Drop columns
df = df.drop(columns=columns_to_drop, errors="ignore")

## 4. Drop Leakage & Non-Informative Columns 
- `actual_qty`, `production_gap_qty` - mathematically derived from the target itself
- `working_hours`, `total_duration_per_shoe_mins` - recorded **after** production completes 
- `machine_capacity_per_day` - constant across all rows (zero variance)
- `item_name` - 370 unique values, most appearing only a few times (too sparse to encode)
- `rate` - dropped from the feature set


In [4]:
# Columns to remove
columns_to_drop = [
    "production_gap_qty",
    "machine_capacity_per_day",
    "item_name",
    "rate",
    "working_hours",
    "total_duration_per_shoe_mins"
]

# Drop columns
df = df.drop(columns=columns_to_drop, errors="ignore")


In [5]:
df.head()

,financial_month,week_of_month,factory_unit,product_family,size_id,grade,planned_qty,actual_qty,no_of_workers,machine_count,bottleneck_flag,order_category
0,Apr,3,Factory Unit 01,Ladies,12,A,35,32,128,8,False,Others
1,Apr,3,Factory Unit 01,Ladies,7,A,41,30,146,7,False,Others
2,Apr,3,Factory Unit 01,Ladies,9,A,15,10,151,8,False,Others
3,Apr,3,Factory Unit 01,Gents,13,A,5,1,128,7,False,Forces/Police
4,Apr,3,Factory Unit 01,Gents,13,A,11,9,103,8,False,Forces/Police


## 5. Exclude Factory Unit 03 
Factory Unit 03 has only 49 of 9,495 rows - too few to support a reliable per-unit signal, especially once a chronological train/test split is applied. The study is scoped to Factory Units 01 and 02 only.


In [6]:
df = df[df['factory_unit'] != 'Factory Unit 03']

## 6. Standardize Factory Unit Label
Normalize `"Factory Unit 02 - LGF"` to `"Factory Unit 02"` for consistent encoding.

In [7]:
df['factory_unit'] = df['factory_unit'].replace(
    'Factory Unit 02 - LGF',
    'Factory Unit 02'
)

Check the resulting row counts per factory unit after exclusion and relabeling.


In [8]:
df['factory_unit'].value_counts()

factory_unit
Factory Unit 01    8921
Factory Unit 02     525
Name: count, dtype: int64

## 7. Encode Bottleneck Flag
Confirmed known **before** production (scheduled machine issues / known process constraints), so it is retained and encoded as a binary feature (0/1).


In [9]:
df["bottleneck_flag"] = df["bottleneck_flag"].map({
    False: 0,
    True: 1
})

In [10]:
df.head()

,financial_month,week_of_month,factory_unit,product_family,size_id,grade,planned_qty,actual_qty,no_of_workers,machine_count,bottleneck_flag,order_category
0,Apr,3,Factory Unit 01,Ladies,12,A,35,32,128,8,0,Others
1,Apr,3,Factory Unit 01,Ladies,7,A,41,30,146,7,0,Others
2,Apr,3,Factory Unit 01,Ladies,9,A,15,10,151,8,0,Others
3,Apr,3,Factory Unit 01,Gents,13,A,5,1,128,7,0,Forces/Police
4,Apr,3,Factory Unit 01,Gents,13,A,11,9,103,8,0,Forces/Police


## 8. Encode Grade (Ordinal)
`A → 1`, `B → 0`.


In [11]:
df["grade"] = df["grade"].map({
    "A": 1,
    "B": 0
})

In [85]:
df.head()

,financial_month,week_of_month,factory_unit,product_family,size_id,grade,planned_qty,no_of_workers,machine_count,bottleneck_flag,order_category,achievement_ratio
0,Apr,3,Factory Unit 01,Ladies,12,1,35,128,8,0,Others,0.914286
1,Apr,3,Factory Unit 01,Ladies,7,1,41,146,7,0,Others,0.731707
2,Apr,3,Factory Unit 01,Ladies,9,1,15,151,8,0,Others,0.666667
3,Apr,3,Factory Unit 01,Gents,13,1,5,128,7,0,Forces/Police,0.200000
4,Apr,3,Factory Unit 01,Gents,13,1,11,103,8,0,Forces/Police,0.818182


## 9. One-Hot Encode Order Category


In [12]:
df = pd.get_dummies(
    df,
    columns=["order_category"],
    drop_first=True,
    dtype=int
)

In [13]:
df.head()

,financial_month,week_of_month,factory_unit,product_family,size_id,grade,planned_qty,actual_qty,no_of_workers,machine_count,bottleneck_flag,order_category_Others,order_category_School
0,Apr,3,Factory Unit 01,Ladies,12,1,35,32,128,8,0,1,0
1,Apr,3,Factory Unit 01,Ladies,7,1,41,30,146,7,0,1,0
2,Apr,3,Factory Unit 01,Ladies,9,1,15,10,151,8,0,1,0
3,Apr,3,Factory Unit 01,Gents,13,1,5,1,128,7,0,0,0
4,Apr,3,Factory Unit 01,Gents,13,1,11,9,103,8,0,0,0


## 10. One-Hot Encode Factory Unit


In [14]:
df = pd.get_dummies(
    df,
    columns=["factory_unit"],
    drop_first=True,
    dtype=int
)

In [15]:
df.head()

,financial_month,week_of_month,product_family,size_id,grade,planned_qty,actual_qty,no_of_workers,machine_count,bottleneck_flag,order_category_Others,order_category_School,factory_unit_Factory Unit 02
0,Apr,3,Ladies,12,1,35,32,128,8,0,1,0,0
1,Apr,3,Ladies,7,1,41,30,146,7,0,1,0,0
2,Apr,3,Ladies,9,1,15,10,151,8,0,1,0,0
3,Apr,3,Gents,13,1,5,1,128,7,0,0,0,0
4,Apr,3,Gents,13,1,11,9,103,8,0,0,0,0


In [16]:
df.tail()

,financial_month,week_of_month,product_family,size_id,grade,planned_qty,actual_qty,no_of_workers,machine_count,bottleneck_flag,order_category_Others,order_category_School,factory_unit_Factory Unit 02
9490,Mar,5,Gents,13,1,8,7,94,8,0,1,0,0
9491,Mar,5,Gents,6,1,39,33,93,8,0,1,0,0
9492,Mar,5,Gents,8,1,9,9,93,7,0,1,0,0
9493,Mar,5,Gents,13,1,46,44,4,8,0,1,0,1
9494,Mar,5,Gents,8,1,5,1,5,7,0,1,0,1


## 11. Inspect Machine Count vs. Achievement Ratio
Check whether `machine_count` carries meaningful separation in mean/median achievement ratio before deciding whether to keep it.


In [18]:
print(df.groupby("machine_count")["actual_qty"].agg(
    ["count", "mean", "median", "std"]
))

               count       mean  median         std
machine_count                                      
7               4161  33.801490     9.0  112.829618
8               5285  30.220057     9.0  100.580043


Dropped - grouped means/medians showed limited explanatory value relative to the other operational features.


In [19]:
df = df.drop(columns=["machine_count"])

In [20]:
df.head()

,financial_month,week_of_month,product_family,size_id,grade,planned_qty,actual_qty,no_of_workers,bottleneck_flag,order_category_Others,order_category_School,factory_unit_Factory Unit 02
0,Apr,3,Ladies,12,1,35,32,128,0,1,0,0
1,Apr,3,Ladies,7,1,41,30,146,0,1,0,0
2,Apr,3,Ladies,9,1,15,10,151,0,1,0,0
3,Apr,3,Gents,13,1,5,1,128,0,0,0,0
4,Apr,3,Gents,13,1,11,9,103,0,0,0,0


## 12. One-Hot Encode Product Family


In [21]:
df = pd.get_dummies(
    df,
    columns=["product_family"],
    drop_first=True,
    dtype=int
)

In [22]:
df.head()

,financial_month,week_of_month,size_id,grade,planned_qty,actual_qty,no_of_workers,bottleneck_flag,order_category_Others,order_category_School,factory_unit_Factory Unit 02,product_family_Ladies
0,Apr,3,12,1,35,32,128,0,1,0,0,1
1,Apr,3,7,1,41,30,146,0,1,0,0,1
2,Apr,3,9,1,15,10,151,0,1,0,0,1
3,Apr,3,13,1,5,1,128,0,0,0,0,0
4,Apr,3,13,1,11,9,103,0,0,0,0,0


## 13. One-Hot Encode Financial Month & Week of Month
Captures seasonality directly through categorical indicators for each month/week-of-month.


In [23]:
df = pd.get_dummies(
    df,
    columns=["financial_month", "week_of_month"],
    drop_first=True,
    dtype=int
)

In [24]:
df.head()

,size_id,grade,planned_qty,actual_qty,no_of_workers,bottleneck_flag,order_category_Others,order_category_School,factory_unit_Factory Unit 02,product_family_Ladies,...,financial_month_Jun,financial_month_Mar,financial_month_May,financial_month_Nov,financial_month_Oct,financial_month_Sep,week_of_month_2,week_of_month_3,week_of_month_4,week_of_month_5
0,12,1,35,32,128,0,1,0,0,1,...,0,0,0,0,0,0,0,1,0,0
1,7,1,41,30,146,0,1,0,0,1,...,0,0,0,0,0,0,0,1,0,0
2,9,1,15,10,151,0,1,0,0,1,...,0,0,0,0,0,0,0,1,0,0
3,13,1,5,1,128,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,13,1,11,9,103,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


## 14. One-Hot Encode Size ID


In [25]:
df = pd.get_dummies(
    df,
    columns=["size_id"],
    drop_first=True,
    dtype=int
)

In [26]:
df.head()

,grade,planned_qty,actual_qty,no_of_workers,bottleneck_flag,order_category_Others,order_category_School,factory_unit_Factory Unit 02,product_family_Ladies,financial_month_Aug,...,week_of_month_3,week_of_month_4,week_of_month_5,size_id_7,size_id_8,size_id_9,size_id_10,size_id_11,size_id_12,size_id_13
0,1,35,32,128,0,1,0,0,1,0,...,1,0,0,0,0,0,0,0,1,0
1,1,41,30,146,0,1,0,0,1,0,...,1,0,0,1,0,0,0,0,0,0
2,1,15,10,151,0,1,0,0,1,0,...,1,0,0,0,0,1,0,0,0,0
3,1,5,1,128,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
4,1,11,9,103,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1


## 15. Final Schema Check
Confirm all columns are numeric and ready for modeling.


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9446 entries, 0 to 9494
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   grade                         9446 non-null   int64
 1   planned_qty                   9446 non-null   int64
 2   actual_qty                    9446 non-null   int64
 3   no_of_workers                 9446 non-null   int64
 4   bottleneck_flag               9446 non-null   int64
 5   order_category_Others         9446 non-null   int64
 6   order_category_School         9446 non-null   int64
 7   factory_unit_Factory Unit 02  9446 non-null   int64
 8   product_family_Ladies         9446 non-null   int64
 9   financial_month_Aug           9446 non-null   int64
 10  financial_month_Dec           9446 non-null   int64
 11  financial_month_Feb           9446 non-null   int64
 12  financial_month_Jan           9446 non-null   int64
 13  financial_month_Jul           9446 non

## 16. Reorder Target Column
Move `achievement_ratio` to the final column position for clarity.


In [28]:
col = df.pop('actual_qty')
df['actual_qty'] = col

In [29]:
df.head()

,grade,planned_qty,no_of_workers,bottleneck_flag,order_category_Others,order_category_School,factory_unit_Factory Unit 02,product_family_Ladies,financial_month_Aug,financial_month_Dec,...,week_of_month_4,week_of_month_5,size_id_7,size_id_8,size_id_9,size_id_10,size_id_11,size_id_12,size_id_13,actual_qty
0,1,35,128,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,32
1,1,41,146,0,1,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,30
2,1,15,151,0,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,10
3,1,5,128,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
4,1,11,103,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,9


## 17. Export Training-Ready Dataset

In [30]:
# Save
df.to_csv("training_ready.csv", index=False)

print("Processed data saved successfully!")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

Processed data saved successfully!
Rows: 9446
Columns: 31
